# 🔷 Delta Lake com Apache Spark

Este notebook demonstra o uso do **Delta Lake** integrado ao **Apache Spark (PySpark)** com um cenário de **E-commerce**.

## 📦 Cenário: E-commerce

Tabelas utilizadas:
- `clientes` — cadastro de clientes
- `produtos` — catálogo de produtos
- `pedidos` — pedidos realizados

## 🗺️ Modelo ER

```
+------------+       +------------+       +------------+
|  clientes  |       |  pedidos   |       |  produtos  |
+------------+       +------------+       +------------+
| id (PK)    |<------| cliente_id |       | id (PK)    |
| nome       |       | id (PK)    |------>| nome       |
| email      |       | produto_id |       | categoria  |
| cidade     |       | quantidade |       | preco      |
| ativo      |       | status     |       | estoque    |
+------------+       | total      |       +------------+
                     | data       |
                     +------------+
```

## 📋 DDL das Tabelas

```sql
-- Tabela clientes
CREATE TABLE clientes (
    id        INT,
    nome      STRING,
    email     STRING,
    cidade    STRING,
    ativo     BOOLEAN
) USING DELTA;

-- Tabela produtos
CREATE TABLE produtos (
    id        INT,
    nome      STRING,
    categoria STRING,
    preco     DOUBLE,
    estoque   INT
) USING DELTA;

-- Tabela pedidos
CREATE TABLE pedidos (
    id          INT,
    cliente_id  INT,
    produto_id  INT,
    quantidade  INT,
    status      STRING,
    total       DOUBLE,
    data        STRING
) USING DELTA;
```

## 1️⃣ Configuração da Sessão Spark com Delta Lake

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Configuração do builder com suporte ao Delta Lake
builder = (
    SparkSession.builder
    .appName("Delta Lake - E-commerce")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "./data/delta/warehouse")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Spark versão: {spark.version}")
print(f"✅ Sessão iniciada com sucesso!")

26/05/03 04:21:19 WARN Utils: Your hostname, DESKTOP-U7BD7UT resolves to a loopback address: 127.0.1.1; using 172.31.143.118 instead (on interface eth0)
26/05/03 04:21:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/joao/spark-lakehouse/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/joao/.ivy2/cache
The jars for the packages stored in: /home/joao/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dd26122f-87b3-4d64-aae3-22ed5f158ad4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (915ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (57ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (99ms)
:: resolution report :: resolve 1808ms :: artif

✅ Spark versão: 3.5.1
✅ Sessão iniciada com sucesso!


## 2️⃣ INSERT — Criando e Populando as Tabelas Delta

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# ── Tabela CLIENTES ──────────────────────────────────────────
schema_clientes = StructType([
    StructField("id",     IntegerType(), False),
    StructField("nome",   StringType(),  True),
    StructField("email",  StringType(),  True),
    StructField("cidade", StringType(),  True),
    StructField("ativo",  BooleanType(), True),
])

dados_clientes = [
    (1, "Ana Silva",    "ana@email.com",    "São Paulo",       True),
    (2, "Bruno Costa",  "bruno@email.com",  "Rio de Janeiro",  True),
    (3, "Carla Souza",  "carla@email.com",  "Curitiba",        True),
    (4, "Diego Lima",   "diego@email.com",  "Florianópolis",   True),
    (5, "Elena Martins","elena@email.com",  "Porto Alegre",    True),
]

df_clientes = spark.createDataFrame(dados_clientes, schema_clientes)
(
    df_clientes.write
    .format("delta")
    .mode("overwrite")
    .save("./data/delta/clientes")
)
print("✅ Tabela 'clientes' criada com sucesso!")
df_clientes.show()

✅ Tabela 'clientes' criada com sucesso!
+---+-------------+---------------+--------------+-----+
| id|         nome|          email|        cidade|ativo|
+---+-------------+---------------+--------------+-----+
|  1|    Ana Silva|  ana@email.com|     São Paulo| true|
|  2|  Bruno Costa|bruno@email.com|Rio de Janeiro| true|
|  3|  Carla Souza|carla@email.com|      Curitiba| true|
|  4|   Diego Lima|diego@email.com| Florianópolis| true|
|  5|Elena Martins|elena@email.com|  Porto Alegre| true|
+---+-------------+---------------+--------------+-----+



In [3]:
# ── Tabela PRODUTOS ──────────────────────────────────────────
schema_produtos = StructType([
    StructField("id",        IntegerType(), False),
    StructField("nome",      StringType(),  True),
    StructField("categoria", StringType(),  True),
    StructField("preco",     DoubleType(),  True),
    StructField("estoque",   IntegerType(), True),
])

dados_produtos = [
    (1, "Notebook Dell",    "Eletrônicos",  3500.00, 15),
    (2, "Mouse Logitech",   "Periféricos",    89.90, 100),
    (3, "Teclado Mecânico", "Periféricos",   250.00,  50),
    (4, "Monitor 24'",      "Eletrônicos",  1200.00,  20),
    (5, "Headset Gamer",    "Periféricos",   350.00,  35),
]

df_produtos = spark.createDataFrame(dados_produtos, schema_produtos)
(
    df_produtos.write
    .format("delta")
    .mode("overwrite")
    .save("./data/delta/produtos")
)
print("✅ Tabela 'produtos' criada com sucesso!")
df_produtos.show()

✅ Tabela 'produtos' criada com sucesso!
+---+----------------+-----------+------+-------+
| id|            nome|  categoria| preco|estoque|
+---+----------------+-----------+------+-------+
|  1|   Notebook Dell|Eletrônicos|3500.0|     15|
|  2|  Mouse Logitech|Periféricos|  89.9|    100|
|  3|Teclado Mecânico|Periféricos| 250.0|     50|
|  4|     Monitor 24'|Eletrônicos|1200.0|     20|
|  5|   Headset Gamer|Periféricos| 350.0|     35|
+---+----------------+-----------+------+-------+



In [4]:
# ── Tabela PEDIDOS ───────────────────────────────────────────
schema_pedidos = StructType([
    StructField("id",          IntegerType(), False),
    StructField("cliente_id",  IntegerType(), True),
    StructField("produto_id",  IntegerType(), True),
    StructField("quantidade",  IntegerType(), True),
    StructField("status",      StringType(),  True),
    StructField("total",       DoubleType(),  True),
    StructField("data",        StringType(),  True),
])

dados_pedidos = [
    (1, 1, 1, 1, "pendente",    3500.00, "2024-01-10"),
    (2, 2, 2, 3, "aprovado",     269.70, "2024-01-11"),
    (3, 3, 3, 1, "pendente",     250.00, "2024-01-12"),
    (4, 4, 4, 2, "entregue",    2400.00, "2024-01-13"),
    (5, 5, 5, 1, "pendente",     350.00, "2024-01-14"),
]

df_pedidos = spark.createDataFrame(dados_pedidos, schema_pedidos)
(
    df_pedidos.write
    .format("delta")
    .mode("overwrite")
    .save("./data/delta/pedidos")
)
print("✅ Tabela 'pedidos' criada com sucesso!")
df_pedidos.show()

✅ Tabela 'pedidos' criada com sucesso!
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|pendente|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|pendente| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|pendente| 350.0|2024-01-14|
+---+----------+----------+----------+--------+------+----------+



In [5]:
# ── INSERT de novos registros via MERGE (upsert) ─────────────
from delta.tables import DeltaTable

novos_pedidos = [
    (6, 1, 2, 2, "aprovado",  179.80, "2024-01-15"),
    (7, 3, 5, 1, "pendente",  350.00, "2024-01-15"),
]

df_novos = spark.createDataFrame(novos_pedidos, schema_pedidos)
delta_pedidos = DeltaTable.forPath(spark, "./data/delta/pedidos")

(
    delta_pedidos.alias("destino")
    .merge(
        df_novos.alias("origem"),
        "destino.id = origem.id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("✅ Novos pedidos inseridos via MERGE!")
spark.read.format("delta").load("./data/delta/pedidos").orderBy("id").show()

✅ Novos pedidos inseridos via MERGE!


+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|pendente|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|pendente| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|pendente| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|pendente| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



## 3️⃣ UPDATE — Atualizando Registros

In [6]:
from delta.tables import DeltaTable

# ── UPDATE: Aprovar pedidos pendentes ────────────────────────
delta_pedidos = DeltaTable.forPath(spark, "./data/delta/pedidos")

delta_pedidos.update(
    condition="status = 'pendente'",
    set={"status": "'aprovado'"}
)

print("✅ Todos os pedidos 'pendente' foram atualizados para 'aprovado'!")
spark.read.format("delta").load("./data/delta/pedidos").orderBy("id").show()

✅ Todos os pedidos 'pendente' foram atualizados para 'aprovado'!


+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



In [7]:
# ── UPDATE: Reajuste de preço nos produtos Eletrônicos (+10%) ─
delta_produtos = DeltaTable.forPath(spark, "./data/delta/produtos")

delta_produtos.update(
    condition="categoria = 'Eletrônicos'",
    set={"preco": "preco * 1.10"}
)

print("✅ Preços de Eletrônicos reajustados em +10%!")
spark.read.format("delta").load("./data/delta/produtos").show()

✅ Preços de Eletrônicos reajustados em +10%!
+---+----------------+-----------+------------------+-------+
| id|            nome|  categoria|             preco|estoque|
+---+----------------+-----------+------------------+-------+
|  3|Teclado Mecânico|Periféricos|             250.0|     50|
|  2|  Mouse Logitech|Periféricos|              89.9|    100|
|  5|   Headset Gamer|Periféricos|             350.0|     35|
|  1|   Notebook Dell|Eletrônicos|3850.0000000000005|     15|
|  4|     Monitor 24'|Eletrônicos|            1320.0|     20|
+---+----------------+-----------+------------------+-------+



In [8]:
# ── UPDATE: Desativar cliente por e-mail ─────────────────────
delta_clientes = DeltaTable.forPath(spark, "./data/delta/clientes")

delta_clientes.update(
    condition="email = 'diego@email.com'",
    set={"ativo": "false"}
)

print("✅ Cliente 'Diego Lima' desativado!")
spark.read.format("delta").load("./data/delta/clientes").show()

✅ Cliente 'Diego Lima' desativado!


+---+-------------+---------------+--------------+-----+
| id|         nome|          email|        cidade|ativo|
+---+-------------+---------------+--------------+-----+
|  5|Elena Martins|elena@email.com|  Porto Alegre| true|
|  2|  Bruno Costa|bruno@email.com|Rio de Janeiro| true|
|  4|   Diego Lima|diego@email.com| Florianópolis|false|
|  3|  Carla Souza|carla@email.com|      Curitiba| true|
|  1|    Ana Silva|  ana@email.com|     São Paulo| true|
+---+-------------+---------------+--------------+-----+



## 4️⃣ DELETE — Removendo Registros

In [9]:
# ── DELETE: Remover pedidos com status 'entregue' ────────────
delta_pedidos = DeltaTable.forPath(spark, "./data/delta/pedidos")

delta_pedidos.delete(condition="status = 'entregue'")

print("✅ Pedidos com status 'entregue' removidos!")
spark.read.format("delta").load("./data/delta/pedidos").orderBy("id").show()

✅ Pedidos com status 'entregue' removidos!
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



In [10]:
# ── DELETE: Remover produto sem estoque ──────────────────────
delta_produtos = DeltaTable.forPath(spark, "./data/delta/produtos")

# Primeiro zeramos o estoque de um produto para simular
delta_produtos.update(
    condition="id = 3",
    set={"estoque": "0"}
)

# Agora deletamos produtos sem estoque
delta_produtos.delete(condition="estoque = 0")

print("✅ Produtos sem estoque removidos!")
spark.read.format("delta").load("./data/delta/produtos").show()

✅ Produtos sem estoque removidos!
+---+--------------+-----------+------------------+-------+
| id|          nome|  categoria|             preco|estoque|
+---+--------------+-----------+------------------+-------+
|  2|Mouse Logitech|Periféricos|              89.9|    100|
|  5| Headset Gamer|Periféricos|             350.0|     35|
|  1| Notebook Dell|Eletrônicos|3850.0000000000005|     15|
|  4|   Monitor 24'|Eletrônicos|            1320.0|     20|
+---+--------------+-----------+------------------+-------+



## 5️⃣ Time Travel — Consultando Versões Anteriores

In [11]:
# ── Histórico de versões da tabela pedidos ───────────────────
delta_pedidos = DeltaTable.forPath(spark, "./data/delta/pedidos")
print("📜 Histórico de operações na tabela 'pedidos':")
delta_pedidos.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

📜 Histórico de operações na tabela 'pedidos':
+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                   |
+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |2026-05-03 04:24:06.792|DELETE   |{predicate -> ["(status#9514 = entregue)"]}                                                                                                           |
|2      |2026-05-03 04:22:51.36 |UPDATE   |{predicate -> ["(status#3832 = pendente)"]}                                                                                        

In [12]:
# ── Lendo a versão 0 (estado inicial) ───────────────────────
print("🕐 Estado inicial da tabela 'pedidos' (versão 0):")
spark.read.format("delta").option("versionAsOf", 0).load("./data/delta/pedidos").show()

🕐 Estado inicial da tabela 'pedidos' (versão 0):
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  5|         5|         5|         1|pendente| 350.0|2024-01-14|
|  3|         3|         3|         1|pendente| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  1|         1|         1|         1|pendente|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
+---+----------+----------+----------+--------+------+----------+



In [13]:
# ── Estado atual da tabela ───────────────────────────────────
print("✅ Estado atual da tabela 'pedidos':")
spark.read.format("delta").load("./data/delta/pedidos").orderBy("id").show()

✅ Estado atual da tabela 'pedidos':
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



## 6️⃣ Schema Evolution — Adicionando uma Nova Coluna

In [14]:
from pyspark.sql.functions import lit

# Adiciona coluna 'desconto' à tabela pedidos com mergeSchema
df_com_desconto = spark.read.format("delta").load("./data/delta/pedidos") \
    .withColumn("desconto", lit(0.0))

(
    df_com_desconto.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save("./data/delta/pedidos")
)

print("✅ Coluna 'desconto' adicionada via Schema Evolution!")
spark.read.format("delta").load("./data/delta/pedidos").show()

✅ Coluna 'desconto' adicionada via Schema Evolution!
+---+----------+----------+----------+--------+------+----------+--------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|desconto|
+---+----------+----------+----------+--------+------+----------+--------+
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|     0.0|
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|     0.0|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|     0.0|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|     0.0|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|     0.0|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|     0.0|
+---+----------+----------+----------+--------+------+----------+--------+



## 7️⃣ Análise Final — Pedidos por Cliente

In [15]:
from pyspark.sql.functions import sum as _sum, count

df_pedidos_final  = spark.read.format("delta").load("./data/delta/pedidos")
df_clientes_final = spark.read.format("delta").load("./data/delta/clientes")

resultado = (
    df_pedidos_final
    .join(df_clientes_final, df_pedidos_final.cliente_id == df_clientes_final.id)
    .groupBy(df_clientes_final.nome)
    .agg(
        count("*").alias("qtd_pedidos"),
        _sum("total").alias("total_gasto")
    )
    .orderBy("total_gasto", ascending=False)
)

print("📊 Resumo de pedidos por cliente:")
resultado.show()

📊 Resumo de pedidos por cliente:
+-------------+-----------+-----------+
|         nome|qtd_pedidos|total_gasto|
+-------------+-----------+-----------+
|    Ana Silva|          2|     3679.8|
|  Carla Souza|          2|      600.0|
|Elena Martins|          1|      350.0|
|  Bruno Costa|          1|      269.7|
+-------------+-----------+-----------+



In [16]:
# Encerrar a sessão Spark
spark.stop()
print("🔴 Sessão Spark encerrada.")

🔴 Sessão Spark encerrada.
